In [1]:
# we'll try to train the model with the NWD loss function for box regression and class prediction.
# The NWD loss is expected to improve the performance of the model, especially in scenarios where
# the bounding boxes vary significantly in size and aspect ratio.
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import models.deformable_detr as detr
from models.matcher import HungarianMatcher
from util.box_ops import NWD
from util.misc import NestedTensor, collate_fn
import  datasets.coco as coco
import numpy as np
import time
import os


/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_args_parser():
    parser = argparse.ArgumentParser('Deformable DETR Detector', add_help=False)
    parser.add_argument('--lr', default=2e-4, type=float)
    parser.add_argument('--lr_backbone_names', default=["backbone.0"], type=str, nargs='+')
    parser.add_argument('--lr_backbone', default=2e-5, type=float)
    parser.add_argument('--lr_linear_proj_names', default=['reference_points', 'sampling_offsets'], type=str, nargs='+')
    parser.add_argument('--lr_linear_proj_mult', default=0.1, type=float)
    parser.add_argument('--batch_size', default=2, type=int)
    parser.add_argument('--weight_decay', default=1e-4, type=float)
    parser.add_argument('--epochs', default=50, type=int)
    parser.add_argument('--lr_drop', default=40, type=int)
    parser.add_argument('--lr_drop_epochs', default=None, type=int, nargs='+')
    parser.add_argument('--clip_max_norm', default=0.1, type=float,
                        help='gradient clipping max norm')


    parser.add_argument('--sgd', action='store_true')

    # Variants of Deformable DETR
    parser.add_argument('--with_box_refine', default=False, action='store_true')
    parser.add_argument('--two_stage', default=False, action='store_true')

    # Model parameters
    parser.add_argument('--frozen_weights', type=str, default=None,
                        help="Path to the pretrained model. If set, only the mask head will be trained")

    # * Backbone
    parser.add_argument('--backbone', default='resnet50', type=str,
                        help="Name of the convolutional backbone to use")
    parser.add_argument('--dilation', action='store_true',
                        help="If true, we replace stride with dilation in the last convolutional block (DC5)")
    parser.add_argument('--position_embedding', default='sine', type=str, choices=('sine', 'learned'),
                        help="Type of positional embedding to use on top of the image features")
    parser.add_argument('--position_embedding_scale', default=2 * np.pi, type=float,
                        help="position / size * scale")
    parser.add_argument('--num_feature_levels', default=4, type=int, help='number of feature levels')

    # * Transformer
    #TODO: Change these
    parser.add_argument('--enc_layers', default=1, type=int,
                        help="Number of encoding layers in the transformer")
    parser.add_argument('--dec_layers', default=1, type=int,
                        help="Number of decoding layers in the transformer")
    parser.add_argument('--dim_feedforward', default=1024, type=int,
                        help="Intermediate size of the feedforward layers in the transformer blocks")
    parser.add_argument('--hidden_dim', default=256, type=int,
                        help="Size of the embeddings (dimension of the transformer)")
    parser.add_argument('--dropout', default=0.1, type=float,
                        help="Dropout applied in the transformer")
    parser.add_argument('--nheads', default=2, type=int,
                        help="Number of attention heads inside the transformer's attentions")
    parser.add_argument('--num_queries', default=300, type=int,
                        help="Number of query slots")
    parser.add_argument('--dec_n_points', default=4, type=int)
    parser.add_argument('--enc_n_points', default=4, type=int)

    # * Segmentation
    parser.add_argument('--masks', action='store_true',
                        help="Train segmentation head if the flag is provided")

    # Loss
    parser.add_argument('--no_aux_loss', dest='aux_loss', action='store_false',
                        help="Disables auxiliary decoding losses (loss at each layer)")

    # * Matcher
    parser.add_argument('--set_cost_class', default=2, type=float,
                        help="Class coefficient in the matching cost")
    parser.add_argument('--set_cost_bbox', default=5, type=float,
                        help="L1 box coefficient in the matching cost")
    parser.add_argument('--set_cost_giou', default=2, type=float,
                        help="giou box coefficient in the matching cost")
    parser.add_argument('--set_cost_nwd', default=7, type=float,
                        help="nwd box coefficient in the matching cost")

    # * Loss coefficients
    parser.add_argument('--mask_loss_coef', default=1, type=float)
    parser.add_argument('--dice_loss_coef', default=1, type=float)
    parser.add_argument('--cls_loss_coef', default=2, type=float)
    parser.add_argument('--bbox_loss_coef', default=5, type=float)
    parser.add_argument('--giou_loss_coef', default=2, type=float)
    parser.add_argument('--focal_alpha', default=0.25, type=float)
    parser.add_argument('--bbox_loss_nwd', default=7, type=float)


    # dataset parameters
    parser.add_argument('--dataset_file', default='coco')
    parser.add_argument('--coco_path', default='./data/coco', type=str)
    parser.add_argument('--coco_panoptic_path', type=str)
    parser.add_argument('--remove_difficult', action='store_true')

    parser.add_argument('--output_dir', default='',
                        help='path where to save, empty for no saving')
    parser.add_argument('--device', default='cuda',
                        help='device to use for training / testing')
    parser.add_argument('--seed', default=42, type=int)
    parser.add_argument('--resume', default='', help='resume from checkpoint')
    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    parser.add_argument('--eval', action='store_true')
    parser.add_argument('--num_workers', default=2, type=int)
    parser.add_argument('--cache_mode', default=False, action='store_true', help='whether to cache images on memory')

    return parser

In [ ]:
#let's start by defining the detr model and run a simple training loop for one epoch as a sanity check.


def main(args):
    # Build the model
    model, criterion, postprocessors = detr.deformable_detr.build(args)
    model.to(args.device)

    # Build the matcher
    matcher = HungarianMatcher(args)

    # Build the optimizer
    param_dicts = [
        {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]},
        {
            "params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
            "lr": args.lr_backbone,
        },
    ]
    optimizer = optim.AdamW(param_dicts, lr=args.lr, weight_decay=args.weight_decay)

    # Build the dataset and dataloader
    dataset_train = coco.build(image_set='train', args=args)
    dataset_val = coco.build(image_set='val', args=args)

    data_loader_train = DataLoader(dataset_train, batch_size=args.batch_size, shuffle=True,
                                   num_workers=args.num_workers, collate_fn=collate_fn)
    data_loader_val = DataLoader(dataset_val, batch_size=args.batch_size, shuffle=False,
                                 num_workers=args.num_workers, collate_fn=collate_fn)

    # Training loop for one epoch
    model.train()
    for samples, targets in data_loader_train:
        samples = samples.to(args.device)
        targets = [{k: v.to(args.device) for k, v in t.items()} for t in targets]

        outputs = model(samples)

        loss_dict = criterion(outputs, targets)
        losses = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict.keys() if k in criterion.weight_dict)

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        #Print the loss for monitoring
        print(f"Loss: {losses.item()}")

    print("Training completed for one epoch.")

In [3]:
#Define the arguments to per parsed and start the main function

parser = argparse.ArgumentParser('Deformable DETR training and evaluation script', parents=[get_args_parser()])
args = parser.parse_args([])


In [5]:
# Build the model
model, criterion, postprocessors = detr.build(args)
model.to(args.device)

# Build the matcher
matcher = HungarianMatcher(args)

# Build the optimizer
param_dicts = [
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]},
    {
        "params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
        "lr": args.lr_backbone,
    },
]
optimizer = optim.AdamW(param_dicts, lr=args.lr, weight_decay=args.weight_decay)

# Build the dataset and dataloader
dataset_train = coco.build(image_set='val', args=args)
# dataset_val = build_coco_dataset(image_set='val', args=args)
data_loader_train = DataLoader(dataset_train, batch_size=args.batch_size, shuffle=True,
                                   num_workers=args.num_workers, collate_fn=collate_fn)

/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loading annotations into memory...
Done (t=0.68s)
creating index...
index created!


In [9]:
{"params": [n for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]}

{'params': ['transformer.level_embed',
  'transformer.encoder.layers.0.self_attn.sampling_offsets.weight',
  'transformer.encoder.layers.0.self_attn.sampling_offsets.bias',
  'transformer.encoder.layers.0.self_attn.attention_weights.weight',
  'transformer.encoder.layers.0.self_attn.attention_weights.bias',
  'transformer.encoder.layers.0.self_attn.value_proj.weight',
  'transformer.encoder.layers.0.self_attn.value_proj.bias',
  'transformer.encoder.layers.0.self_attn.output_proj.weight',
  'transformer.encoder.layers.0.self_attn.output_proj.bias',
  'transformer.encoder.layers.0.norm1.weight',
  'transformer.encoder.layers.0.norm1.bias',
  'transformer.encoder.layers.0.linear1.weight',
  'transformer.encoder.layers.0.linear1.bias',
  'transformer.encoder.layers.0.linear2.weight',
  'transformer.encoder.layers.0.linear2.bias',
  'transformer.encoder.layers.0.norm2.weight',
  'transformer.encoder.layers.0.norm2.bias',
  'transformer.decoder.layers.0.cross_attn.sampling_offsets.weight',


In [ ]:
# Training loop for one epoch
model.train()
loss_list = []
index = 0
for samples, targets in data_loader_train:
    samples = samples.to(args.device)
    targets = [{k: v.to(args.device) for k, v in t.items()} for t in targets]

    outputs = model(samples)

    loss_dict = criterion(outputs, targets)
    losses = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict.keys() if k in criterion.weight_dict)

    optimizer.zero_grad()
    losses.backward()
    optimizer.step()

    #Print the loss for monitoring
    loss_list.append(losses.item())
    index += 1
    if index % 10 == 0:
        print(f"Iteration {index}, Loss: {losses.item()}")
print(f"Average Loss: {np.mean(loss_list)}")
print("Training completed for one epoch.")

/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loss: 1.0454198122024536
Loss: 1.0297385454177856
Loss: 0.7620146870613098
Loss: 0.8604828119277954
Loss: 0.9717600345611572
Loss: 0.6968595385551453
Loss: 1.1279577016830444
Loss: 1.0356078147888184
Loss: 0.7130377888679504
Loss: 0.7347352504730225
Loss: 0.8676521182060242
Loss: 1.001650333404541
Loss: 0.5533918142318726
Loss: 0.8612484335899353
Loss: 0.6934999227523804
Loss: 1.2428758144378662
Loss: 0.7631411552429199
Loss: 0.651494026184082
Loss: 0.6433090567588806
Loss: 0.707787036895752
Loss: 0.6787254214286804
Loss: 0.9640060663223267
Loss: 0.8585209846496582
Loss: 0.7567168474197388
Loss: 0.705458402633667
Loss: 0.8193609118461609
Loss: 0.6561691761016846
Loss: 0.9218853116035461
Loss: 0.9060016870498657
Loss: 0.7607517838478088
Loss: 0.8362759947776794
Loss: 0.639055609703064
Loss: 0.6784257292747498
Loss: 0.6781387329101562
Loss: 0.7013454437255859
Loss: 0.8012681007385254
Loss: 0.8587538003921509
Loss: 0.6918239593505859
Loss: 0.7711514234542847
Loss: 0.5286533236503601
Loss:

KeyboardInterrupt: 

In [ ]:
#Clear memory from GPUS
torch.cuda.empty_cache()
del model
del criterion
del optimizer
del data_loader_train


: 